In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import pickle
from time import time

from utils import DEFAULT_DATA, get_real_muons, calc_pt

In [ ]:
MUONS_DATA = DEFAULT_DATA        # flat table: one dataset per column, a weight per row
NUM_SAMPLES = int(20e6)          # None = the whole file

In [ ]:
# One flat table, so there are no weight groups to walk: the importance weight is a
# column. get_real_muons rescales it when NUM_SAMPLES caps the rows, so the sub-sample
# still carries the full physical rate.
real = get_real_muons(MUONS_DATA, num_samples=NUM_SAMPLES)
pz_real = real[:, 2]
pt_real = calc_pt(real[:, 0], real[:, 1])
weights = real[:, 7]
print(f"{real.shape[0]:,} muons | sum of weights {weights.sum(dtype=np.float64):,.4g}")

In [ ]:
with open('outputs/gmm_model.pkl', 'rb') as f:
    model = pickle.load(f)       # a single weighted GMM (train_gmm.py)
model['gmm'], model['weight']

In [ ]:
n_samples = int(300e6)
gmm, mean, std = model['gmm'], model['mean'], model['std']
t1 = time()
sample = gmm.sample(n_samples)[0] * std + mean
print(f"Sampling {n_samples} samples took {time()-t1:.2f} seconds")
gen_pz, gen_pt = sample[:, 0], sample[:, 1]
del sample
print(f"Generated pz shape: {gen_pz.shape}, pt shape: {gen_pt.shape}")

In [ ]:
from matplotlib.colors import LogNorm

def compare_1d_histograms(real_sample, gen_sample, bins=120, range_vals=None, xlabel="Value", real_w=None):
    """Overlays 1D distributions of real and generated samples."""
    fig, ax = plt.subplots(figsize=(8, 5))
    
    # Auto-detect range if none is provided
    if range_vals is None:
        range_vals = (min(np.min(real_sample), np.min(gen_sample)), 
                      max(np.max(real_sample), np.max(gen_sample)))
        
    b = np.linspace(range_vals[0], range_vals[1], bins)
    
    # Plot marginals
    ax.hist(real_sample, bins=b, weights=real_w, density=True, 
            histtype='step', color='k', label='Real')
    ax.hist(gen_sample, bins=b, density=True, 
            histtype='step', color='C1', label='Generated')
    
    ax.set_xlabel(xlabel, fontsize=14)
    ax.set_ylabel('Density', fontsize=14)
    ax.legend(fontsize=11)
    
    fig.tight_layout()
    return fig, ax


def compare_2d_histograms(real_pz, real_pt, gen_pz, gen_pt, real_w=None):
    """Creates side-by-side 2D Pz-Pt density histograms with a shared color scale."""
    pz_bins = np.linspace(0, 400, 201)
    pt_bins = np.linspace(0, 13, 201)
    
    # Calculate 2D densities
    Hr, _, _ = np.histogram2d(real_pz, real_pt, bins=[pz_bins, pt_bins], 
                              weights=real_w, density=True)
    Hg, _, _ = np.histogram2d(gen_pz, gen_pt, bins=[pz_bins, pt_bins], 
                              density=True)
    
    # Shared LogNorm for accurate visual comparison
    pos_vals = np.concatenate([Hr[Hr > 0].ravel(), Hg[Hg > 0].ravel()])
    norm = LogNorm(vmin=pos_vals.min(), vmax=max(Hr.max(), Hg.max()))
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
    
    # Draw meshes
    for ax, H, title in zip(axes, [Hr, Hg], ['Real', 'Generated']):
        im = ax.pcolormesh(pz_bins, pt_bins, H.T, cmap='viridis', norm=norm)
        ax.set_xlim(0, 400)
        ax.set_ylim(0, 13)
        ax.set_xlabel('$P_z$ [GeV]', fontsize=14)
        ax.set_ylabel('$P_t$ [GeV]', fontsize=14)
        ax.set_title(title, fontsize=14)
        
    # Shared colorbar
    fig.colorbar(im, ax=axes.tolist(), label='Density')
    
    return fig, axes

In [ ]:
compare_2d_histograms(pz_real, pt_real, gen_pz, gen_pt, real_w=weights)